# Intermediate 14 — Logistic Regression Overview

Practice notebook for the **"Logistic Regression Overview"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Why not linear regression for a binary response?

When the response \(Y\) is binary (0/1), a linear model for
\(E[Y \mid X]\) is often inappropriate because it can produce values
outside \([0,1]\) — predictions below 0 or above 1 are not valid
probabilities. **Logistic regression** instead models the probability

$$
\pi(x) = P(Y=1 \mid X=x)
$$

via the **logit** link:

$$
\log\frac{\pi(x)}{1 - \pi(x)} = \beta^\top x.
$$

The logit is the **log-odds**: \(\log(\text{odds}) = \log(\pi/(1-\pi))\).
Solving for \(\pi\) gives the logistic (sigmoid) function

$$
\pi(x) = \frac{\exp(\beta^\top x)}{1 + \exp(\beta^\top x)}
            = \frac{1}{1 + \exp(-\beta^\top x)}.
$$

Let's plot the sigmoid for a few choices of \(\beta\) to build intuition.


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

xs = np.linspace(-6, 6, 400)
fig, ax = plt.subplots(figsize=(9, 5))
for b in [-2.0, -0.5, 0.5, 2.0]:
    ax.plot(xs, sigmoid(b * xs), label=rf"$\sigma({b}\,x)$")
ax.axhline(0.5, color="black", lw=0.8, ls="--", alpha=0.6)
ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.6)
ax.set_xlabel("x"); ax.set_ylabel(r"$\pi(x) = \sigma(\beta^\top x)$")
ax.set_title("Logistic sigmoid for several slopes")
ax.set_ylim(-0.02, 1.02)
ax.legend()
plt.show()

# Sanity: sigmoid maps R -> (0,1), symmetric about 0.5
print("sigmoid(0)  =", sigmoid(0.0))
print("sigmoid(10) =", sigmoid(10.0))
print("sigmoid(-10)=", sigmoid(-10.0))
print("1 - sigmoid(z) == sigmoid(-z):", np.allclose(1 - sigmoid(xs), sigmoid(-xs)))


---
## Part 2 — Fitting by maximum likelihood (IRLS / Newton-Raphson)

Given data \((x_i, y_i)\) with \(y_i \in \{0,1\}\), the conditional
log-likelihood is

$$
\ell(\beta) = \sum_{i=1}^{n} \big[ y_i \log \pi_i + (1 - y_i) \log(1 - \pi_i) \big],
\qquad \pi_i = \sigma(\beta^\top x_i).
$$

The gradient and Hessian are remarkably clean:

$$
\nabla \ell = X^\top (y - \pi),
\qquad
\nabla^2 \ell = -X^\top W X,
$$

where \(W = \mathrm{diag}(\pi_i (1 - \pi_i))\). Newton-Raphson updates
\(\beta\) by

$$
\beta^{(t+1)} = \beta^{(t)} + (X^\top W X)^{-1} X^\top (y - \pi).
$$

This is exactly **Iteratively Reweighted Least Squares (IRLS)**. We'll
implement it from scratch on a simulated separable-ish dataset.


In [ ]:
# Simulate a separable-ish binary dataset with one predictor + intercept
rng = np.random.default_rng(42)
n = 120
x1 = rng.normal(0, 1, size=n)                       # predictor
true_beta = np.array([-1.0, 2.5])                  # [intercept, slope]
X = np.column_stack([np.ones(n), x1])               # design matrix
eta_true = X @ true_beta
pi_true = sigmoid(eta_true)
y = (rng.uniform(size=n) < pi_true).astype(int)    # Bernoulli draws

print(f"n = {n}, true beta = {true_beta}")
print(f"class balance: {y.mean():.3f} (fraction of 1s)")

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x1, y + 0.0*rng.uniform(-0.05, 0.05, size=n),
           alpha=0.5, label="observed $y_i$ (jittered)")
xs = np.linspace(x1.min(), x1.max(), 200)
ax.plot(xs, sigmoid(true_beta[0] + true_beta[1]*xs),
       color="green", lw=2, label=r"true $\pi(x)$")
ax.set_xlabel("x1"); ax.set_ylabel("y / probability")
ax.set_title("Simulated binary data with true logistic curve")
ax.legend()
plt.show()


In [ ]:
def log_likelihood(beta, X, y):
    eta = X @ beta
    # numerically stable log-sigmoid: log(sigmoid(z)) = -log(1+exp(-z))
    # and log(1 - sigmoid(z)) = -z - log(1+exp(-z)) = log(sigmoid(-z))
    log_pi     = -np.logaddexp(0.0, -eta)         # log sigmoid(eta)
    log_one_pi = -np.logaddexp(0.0,  eta)         # log sigmoid(-eta)
    return np.sum(y * log_pi + (1 - y) * log_one_pi)

def irls(X, y, max_iter=100, tol=1e-10, beta0=None):
    n, p = X.shape
    beta = np.zeros(p) if beta0 is None else beta0.copy()
    history = [log_likelihood(beta, X, y)]
    for it in range(max_iter):
        eta = X @ beta
        pi = sigmoid(eta)
        W = pi * (1 - pi)                          # weights
        # gradient X'(y - pi); Hessian -X'WX -> solve (X'WX) d = X'(y-pi)
        XtWX = X.T @ (X * W[:, None])
        g = X.T @ (y - pi)
        d = np.linalg.solve(XtWX + 1e-8*np.eye(p), g)   # tiny ridge for stability
        beta_new = beta + d
        history.append(log_likelihood(beta_new, X, y))
        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new
            break
        beta = beta_new
    return beta, np.array(history)

beta_irls, hist = irls(X, y)
print(f"IRLS beta  = {beta_irls}")
print(f"true beta = {true_beta}")
print(f"iterations: {len(hist)-1}, final log-lik: {hist[-1]:.6f}")

# Log-likelihood should be monotonically increasing (Newton on concave likelihood)
print("log-lik monotone non-decreasing:",
      np.all(np.diff(hist) > -1e-9))


---
## Part 3 — Cross-check against `scipy.optimize.minimize`

As an independent check, we minimize the **negative** log-likelihood with
a generic optimizer. The two fits should agree to numerical tolerance.


In [ ]:
from scipy.optimize import minimize

def neg_log_likelihood(beta, X, y):
    return -log_likelihood(beta, X, y)

res = minimize(neg_log_likelihood, np.zeros(X.shape[1]),
               args=(X, y), method="Newton-CG",
               jac=lambda b, X, y: -(X.T @ (y - sigmoid(X @ b))),
               options={"xtol": 1e-10, "disp": False})
beta_sp = res.x

print(f"IRLS     beta = {beta_irls}")
print(f"scipy    beta = {beta_sp}")
print(f"max abs diff  = {np.max(np.abs(beta_irls - beta_sp)):.2e}")
assert np.allclose(beta_irls, beta_sp, atol=1e-5)
print("IRLS agrees with scipy.optimize ✔")

# Also report the maximized log-likelihood under both
print(f"IRLS  log-lik = {log_likelihood(beta_irls, X, y):.6f}")
print(f"scipy log-lik = {log_likelihood(beta_sp,  X, y):.6f}")


---
## Part 4 — Fitted sigmoid and decision boundary

With \(\hat{\beta}\) in hand we plot the fitted sigmoid over the data.
The **decision boundary** is the point where \(\pi(x) = 0.5\), i.e.
\(\beta^\top x = 0\). For one predictor that is

$$
x_1^* = -\hat{\beta}_0 / \hat{\beta}_1.
$$

We classify \(\hat{y}_i = 1\) when \(\pi_i \geq 0.5\), and report the
training accuracy.


In [ ]:
b0_hat, b1_hat = beta_irls
boundary = -b0_hat / b1_hat

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x1, y + 0.0*rng.uniform(-0.05, 0.05, size=n),
           alpha=0.5, label="observed $y_i$ (jittered)")
xs = np.linspace(x1.min(), x1.max(), 300)
ax.plot(xs, sigmoid(b0_hat + b1_hat*xs), color="crimson", lw=2,
        label=rf"fit $\hat{{\pi}}(x)$: {b0_hat:.3f} + {b1_hat:.3f}$\,x$")
ax.plot(xs, sigmoid(true_beta[0] + true_beta[1]*xs),
        color="green", lw=1.5, ls="--", alpha=0.7, label=r"true $\pi(x)$")
ax.axvline(boundary, color="purple", ls=":", lw=1.5,
           label=rf"boundary $x^*={boundary:.3f}$")
ax.axhline(0.5, color="black", lw=0.8, ls="--", alpha=0.5)
ax.set_xlabel("x1"); ax.set_ylabel("y / probability")
ax.set_title("Fitted logistic regression vs. truth")
ax.legend()
plt.show()

# Classification at 0.5 threshold and training accuracy
pi_hat = sigmoid(X @ beta_irls)
y_pred = (pi_hat >= 0.5).astype(int)
acc = np.mean(y_pred == y)
print(f"Decision boundary x* = {boundary:.4f}")
print(f"Training accuracy    = {acc:.4f}")

# Confusion matrix
TP = np.sum((y_pred == 1) & (y == 1))
TN = np.sum((y_pred == 0) & (y == 0))
FP = np.sum((y_pred == 1) & (y == 0))
FN = np.sum((y_pred == 0) & (y == 1))
print(f"Confusion: TP={TP} TN={TN} FP={FP} FN={FN}")


---
## Part 5 — Interpreting coefficients as log-odds (and odds ratios)

Each coefficient \(\beta_j\) is the change in the **log-odds** per unit
increase in \(x_j\), holding other variables fixed. Exponentiating gives the
**odds ratio**:

$$
\frac{\mathrm{odds}(x_j + 1)}{\mathrm{odds}(x_j)} = \exp(\beta_j).
$$

The PDF's default example: if \(\hat{\beta}_1 = 0.05\), then a one-unit
increase in \(X_1\) multiplies the odds of default by \(\exp(0.05) \approx
1.051\), i.e. about a 5.1% increase. We verify the odds-ratio identity by
direct **perturbation**: compute the odds at \(x\) and at \(x+1\) and take
their ratio — it should equal \(\exp(\hat{\beta}_1)\).


In [ ]:
# Odds ratio from the fitted slope
or_beta = np.exp(b1_hat)
print(f"exp(beta1_hat) = {or_beta:.6f}")

# Verify by perturbation: pick a reference x, compare odds(x) and odds(x+1)
x_ref = 0.0
eta0 = b0_hat + b1_hat * x_ref
eta1 = b0_hat + b1_hat * (x_ref + 1.0)
odds0 = sigmoid(eta0) / (1 - sigmoid(eta0))
odds1 = sigmoid(eta1) / (1 - sigmoid(eta1))
or_perturb = odds1 / odds0
print(f"odds(x={x_ref})     = {odds0:.6f}")
print(f"odds(x={x_ref+1})   = {odds1:.6f}")
print(f"odds ratio by perturbation = {or_perturb:.6f}")
print(f"matches exp(beta1)         = {or_beta:.6f}")
assert np.isclose(or_perturb, or_beta)
print("Odds-ratio identity verified ✔")

# PDF-style statement: a one-unit increase multiplies odds by exp(beta1_hat)
pct = 100 * (or_beta - 1)
print(f"A one-unit increase in x1 changes the odds of Y=1 by {or_beta:.4f}x "
      f"({pct:+.1f}%).")


---
## Reproducing the PDF's default example

The PDF specifies a default-probability model with two predictors
(\(X_1\) = debt-to-income ratio, \(X_2\) = credit score):

$$
\log\frac{\pi}{1-\pi} = \beta_0 + \beta_1 X_1 + \beta_2 X_2.
$$

With \(\hat{\beta}_1 = 0.05\), a one-unit increase in \(X_1\) multiplies
the odds of default by \(\exp(0.05) \approx 1.051\) — about a 5.1% increase
in the odds, holding credit score fixed. We reproduce that number exactly
and visualize the odds-multiplier curve as a function of \(\beta_1\).


In [ ]:
# Reproduce the PDF number directly
beta1_pdf = 0.05
or_pdf = np.exp(beta1_pdf)
print(f"exp(0.05) = {or_pdf:.4f}  -> {100*(or_pdf-1):+.1f}% change in odds")

# How does the odds multiplier grow with beta1?
bs = np.linspace(-0.2, 0.2, 200)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(bs, np.exp(bs), color="navy", lw=2, label=r"$\exp(\beta_1)$")
ax.axhline(1.0, color="black", lw=0.8, ls="--", alpha=0.5)
ax.axvline(0.0, color="black", lw=0.8, ls="--", alpha=0.5)
ax.scatter([beta1_pdf], [or_pdf], color="crimson", zorder=5,
           label=rf"PDF: $\beta_1=0.05 \Rightarrow {or_pdf:.3f}$")
ax.set_xlabel(r"$\beta_1$ (log-odds change per unit $X_1$)")
ax.set_ylabel(r"odds multiplier $\exp(\beta_1)$")
ax.set_title("Odds multiplier vs. coefficient (default example)")
ax.legend()
plt.show()

# Two-predictor simulation reproducing the structure of the PDF example
rng2 = np.random.default_rng(7)
n2 = 400
X1 = rng2.uniform(0, 10, n2)        # debt-to-income
X2 = rng2.normal(650, 80, n2)       # credit score
beta_default = np.array([-3.0, 0.05, -0.004])   # [b0, b1, b2]
Xd = np.column_stack([np.ones(n2), X1, X2])
pi_d = sigmoid(Xd @ beta_default)
y_d = (rng2.uniform(size=n2) < pi_d).astype(int)

beta_d, _ = irls(Xd, y_d)
print(f"Two-predictor fit: beta = {beta_d}")
print(f"exp(beta1_hat)    = {np.exp(beta_d[1]):.4f}  (true 0.05 -> {np.exp(0.05):.4f})")
print(f"exp(beta2_hat)    = {np.exp(beta_d[2]):.4f}  (true exp(-0.004) = {np.exp(-0.004):.4f})")


---
**Your turn:**

- Add a second predictor to the Part-2 simulation and re-fit with IRLS.
  Does the recovered slope match the truth more closely than in 1D?
- Re-run IRLS from a *bad* starting value (e.g. \(\beta^{(0)} = (5, -5)\)).
  Does it still converge to the same optimum? Why?
- Lower the sample size to \(n = 20\) and re-fit many times. How variable is
  \(\hat{\beta}_1\) across replications? Compare to the large-\(n\) case.
- Change the decision threshold from \(0.5\) to \(0.3\) and recompute the
  confusion matrix. Which class gets more false positives, and why does that
  matter for a *default-probability* model?
- Re-derive the gradient \(\nabla\ell = X^\top(y-\pi)\) from the
  log-likelihood by hand, then verify each component numerically with a
  finite-difference check on the simulated dataset.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Write `irls(X, y)` returning \(\hat{\beta}\) and the log-likelihood
trace. Verify that the log-likelihood is non-decreasing across iterations on
the simulated dataset from Part 2.

2. Show that the Newton-Raphson update can be written as a weighted
least-squares step
\(\beta^{(t+1)} = (X^\top W X)^{-1} X^\top W z\)
with **working response** \(z = X\beta^{(t)} + (y-\pi)/W\).
Implement this form and confirm it gives the same \(\hat{\beta}\).

3. For the fitted model, compute the **deviance**
\(D = -2(\ell(\hat{\beta}) - \ell_{\text{sat}})\), where the saturated model
has \(\ell_{\text{sat}} = 0\) for binary data. Verify numerically that
\(D = -2\,\ell(\hat{\beta})\) and that smaller deviance means a better fit.

4. Using the two-predictor default dataset, report the odds ratio for
\(X_1\) (debt-to-income) and \(X_2\) (credit score) and interpret each in a
sentence. Which predictor has the stronger effect *per one unit*? Per one
standard deviation?

5. Implement a finite-difference gradient check: for a random \(\beta\),
compare the analytic gradient \(X^\top(y-\pi)\) to a central
difference of the log-likelihood. Report the max error; it should be
\(\sim 10^{-7}\) or smaller.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** The `irls` function from Part 2 already returns the trace;
just check it is non-decreasing:

```python
beta, hist = irls(X, y)
print(np.all(np.diff(hist) > -1e-9))   # True (Newton on concave likelihood)
```

Because the log-likelihood is concave in \(\beta\), Newton-Raphson with
a step length of 1 always increases it.

**2.** The working-response form of IRLS uses
\(z = X\beta^{(t)} + (y-\pi)/W\) and solves
\((X^\topWX)\beta = X^\topWz\):

```python
def irls_z(X, y, max_iter=100, tol=1e-10):
    n, p = X.shape
    beta = np.zeros(p)
    for _ in range(max_iter):
        pi = sigmoid(X @ beta)
        W = pi * (1 - pi)
        z = X @ beta + (y - pi) / W
        beta_new = np.linalg.solve(X.T @ (X * W[:, None]),
                                   X.T @ (W * z))
        if np.max(np.abs(beta_new - beta)) < tol:
            beta = beta_new; break
        beta = beta_new
    return beta

print(np.allclose(irls_z(X, y), irls(X, y)[0]))   # True
```

**3.** For binary data the saturated log-likelihood is 0 (each \(\pi_i=y_i\)
gives \(\log 1 = 0\)), so the deviance is just \(-2\ell(\hat{\beta})\):

```python
ell_hat = log_likelihood(beta_irls, X, y)
D = -2 * ell_hat
print(D, -2 * ell_hat, np.isclose(D, -2 * ell_hat))
```

Smaller deviance = larger likelihood = better fit.

**4.** Odds ratios are \(\exp(\hat{\beta}_j)\). Per one unit, the predictor
with larger \(|\hat{\beta}_j|\) has the stronger effect; per one standard
deviation, compare \(\exp(\hat{\beta}_j \cdot \mathrm{sd}(X_j))\):

```python
beta_d, _ = irls(Xd, y_d)
sd = Xd[:, 1:].std(axis=0)
print("per unit:        ", np.exp(beta_d[1:]))
print("per 1 sd:        ", np.exp(beta_d[1:] * sd))
```

**5.** Central-difference check on a random \(\beta\):

```python
rng = np.random.default_rng(0)
b = rng.normal(0, 1, X.shape[1])
eps = 1e-6
g_fd = np.array([(log_likelihood(b+eps*e, X, y)
                - log_likelihood(b-eps*e, X, y)) / (2*eps)
                for e in np.eye(X.shape[1])])
g_an = X.T @ (y - sigmoid(X @ b))
print(np.max(np.abs(g_an - g_fd)))   # ~1e-7 or smaller
```


</details>
